# ML Jobs: Custom Docker Images

**Session 3 — Deep Dive: Distributed Training & Hyperparameter Optimization**  
**Notebook 2 of 4** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Dockerfile** | Define a custom training environment with exact dependencies |
| **Image Registry** | Push images to Snowflake's built-in image registry |
| **`submit_directory` + Dockerfile** | Submit a job using your custom image |
| **When to use Docker** | Scenarios where a Dockerfile is preferred over `requirements.txt` |

---

## When to Use a Dockerfile

In Notebook 01 we used `submit_directory` with a `requirements.txt` — Snowflake built the
container for us. That works for most cases. Use a **Dockerfile** when you need:

| Scenario | Why Docker |
|----------|------------|
| **System-level packages** | `apt-get install` for C libraries, CUDA toolkit, etc. |
| **Exact reproducibility** | Pin every layer — OS, Python version, library versions |
| **Pre-built wheels** | Include `.whl` files that can't be pip-installed |
| **Multi-stage builds** | Compile C extensions in a build stage, copy to slim runtime |
| **Org-standard base images** | Start from your company's approved base image |

### Architecture

```
 Local Machine            Snowflake Image Registry        SPCS Compute Pool
 ┌───────────┐            ┌──────────────────┐            ┌─────────────────┐
 │ Dockerfile │  docker   │ <acct>.registry   │  ML Jobs  │  Container from  │
 │ train.py   │  build &  │ .snowflakecomputi │  pulls    │  your image      │
 │ reqs.txt   │  push ──> │ ng.com/<db>/<sch> │  ──────>  │  runs train.py   │
 └───────────┘            │ /my_repo:latest   │           └─────────────────┘
                          └──────────────────┘
```

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import logging
import os

from utils import get_config
from utils import get_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

config = get_config("config.yaml")
DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse
COMPUTE_POOL = config.compute.compute_pool
STAGE = f"{DB}.{SCHEMA}.{config.stages.job_payloads}"

session = get_session(
    connection_name=config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)

print(f"Connected as {session.get_current_user()} | Role: {session.get_current_role()}")
print(f"Context: {DB}.{SCHEMA}")

## 2 | Inspect the Dockerfile

Our Dockerfile is intentionally simple — it demonstrates the pattern without unnecessary complexity.

### Key Design Decisions

| Decision | Rationale |
|----------|-----------|
| `python:3.10-slim` | Matches SPCS runtime version, small image |
| `pip install` in Dockerfile | Dependencies baked into image — no install at job start |
| `COPY` only the entrypoint | Keep image small; data comes from Snowflake tables |
| `ENTRYPOINT` | ML Jobs overrides this, but useful for local testing |

In [ ]:
with open("session3_distributed_training_and_hpo/job_payload/Dockerfile") as f:
    print(f.read())

## 3 | Build & Push the Docker Image

Snowflake provides a built-in **image registry** in every account.
You don't need Docker Hub or ECR — images stay within Snowflake's security perimeter.

### Image Registry URL Format

```
<org>-<account>.registry.snowflakecomputing.com/<database>/<schema>/<image_repo>:<tag>
```

### Prerequisites

1. An **image repository** in your schema
2. Docker installed locally
3. Docker authenticated to Snowflake's registry

The cells below show the commands — run them in your terminal (not in this notebook)
since Docker builds require a local Docker daemon.

In [ ]:
session.sql(f"""
    CREATE IMAGE REPOSITORY IF NOT EXISTS {DB}.{SCHEMA}.ML_TRAINING_IMAGES
    COMMENT = 'Docker images for ML training jobs'
""").collect()

repo_url = session.sql(f"""
    SHOW IMAGE REPOSITORIES LIKE 'ML_TRAINING_IMAGES' IN SCHEMA {DB}.{SCHEMA}
""").collect()[0]["repository_url"]

print(f"Image Repository URL: {repo_url}")
print(f"\nFull image tag: {repo_url}/patient_risk_xgb:latest")

### Docker Commands (run in terminal)

```bash
# 1. Authenticate Docker to Snowflake registry
docker login <org>-<account>.registry.snowflakecomputing.com \
  --username <user> --password <password>

# 2. Build the image
cd session3_distributed_training_and_hpo/job_payload
docker build -t patient_risk_xgb:latest .

# 3. Tag for Snowflake registry
docker tag patient_risk_xgb:latest \
  <repo_url>/patient_risk_xgb:latest

# 4. Push to Snowflake
docker push <repo_url>/patient_risk_xgb:latest
```

Replace `<repo_url>` with the URL printed above.

## 4 | Submit the Job with a Custom Image

Once the image is pushed, `submit_directory` can reference it.
The key difference from Notebook 01 is that dependencies are already
baked into the image — no `pip install` at job start time.

### Performance Benefit

| Approach | Cold Start | Reproducibility |
|----------|-----------|------------------|
| `requirements.txt` | ~30-60s (pip install) | Depends on PyPI state |
| Dockerfile | ~5-10s (image pull) | Fully deterministic |

In [ ]:
from datetime import datetime
from snowflake.ml.feature_store import FeatureStore

fs = FeatureStore(session, DB, SCHEMA, default_warehouse=WH)
fv = fs.get_feature_view("PATIENT_FEATURES", "v1")

spine_df = session.table(f"{DB}.{SCHEMA}.RAW_PATIENT_DATA").select(
    "PATIENT_ID", "TIMESTAMP"
)

DATASET_NAME = "PATIENT_TRAINING_DATASET"
DATASET_VERSION = datetime.now().strftime("v_%Y%m%d_%H%M%S")

ds = fs.generate_dataset(
    name=f"{DB}.{SCHEMA}.{DATASET_NAME}",
    spine_df=spine_df,
    features=[fv],
    spine_timestamp_col="TIMESTAMP",
    version=DATASET_VERSION,
)

print(f"Generated dataset: {DATASET_NAME} / {DATASET_VERSION}")
print(f"  Rows: {ds.read.to_pandas().shape[0]}")

In [ ]:
from snowflake.ml.jobs import submit_directory

JOB_DIR = "session3_distributed_training_and_hpo/job_payload"
IMAGE_PATH = "/ml_demo_pipeline_db/healthcare/ml_training_images/patient_risk_xgb:latest"
ENTRY_SCRIPT = "train_xgb_docker.py"

spec_overrides = {
    "spec": {
        "containers": [
            {
                "name": "main",
                "image": IMAGE_PATH,
            }
        ]
    }
}

job = submit_directory(
    dir_path=JOB_DIR,
    entrypoint=ENTRY_SCRIPT,
    compute_pool=COMPUTE_POOL,
    stage_name=STAGE,
    spec_overrides=spec_overrides,
    target_instances=1,
    env_vars={
        "DATABASE": DB,
        "SCHEMA": SCHEMA,
        "MODEL_NAME": "PATIENT_RISK_XGB_DOCKER",
        "N_ESTIMATORS": "200",
        "MAX_DEPTH": "8",
        "LEARNING_RATE": "0.1",
        "TRAINING_DATASET_NAME": DATASET_NAME,
        "TRAINING_DATASET_VERSION": DATASET_VERSION,
    },
)

print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")

In [ ]:
print(f"Waiting for job {job.id} ...")
job.wait()
print(f"Final status: {job.status}")

logs = job.get_logs()
print("\n" + "=" * 60)
print("JOB LOGS (tail)")
print("=" * 60)
print(logs[-2000:] if len(logs) > 2000 else logs)

## 5 | Verify the Model

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session, database_name=DB, schema_name=SCHEMA)

model = registry.get_model("PATIENT_RISK_XGB_DOCKER")
print(f"Model: {model.name}")

v = model.last()
metrics = v.show_metrics()
print(f"  {v.version_name}")
for k, val in metrics.items():
    print(f"    {k}: {val}")

## 6 | Comparison: `requirements.txt` vs Dockerfile

| Aspect | requirements.txt | Dockerfile |
|--------|-----------------|------------|
| **Setup effort** | Zero — just list packages | Write Dockerfile, build, push |
| **Cold start time** | Slower — pip install at launch | Faster — dependencies pre-baked |
| **Reproducibility** | Good (with pinned versions) | Excellent (entire OS frozen) |
| **System packages** | Not supported | Full control (`apt-get`) |
| **Image size** | Managed by Snowflake | You control it |
| **Best for** | Prototyping, standard ML | Production, custom runtimes |

### Recommendation

- **Start with `requirements.txt`** — it's simpler and works for 90% of cases
- **Upgrade to Dockerfile** when you need system packages, exact reproducibility,
  or faster cold starts in production

## 7 | Summary

In this notebook we:

1. **Examined** a Dockerfile that packages XGBoost training into a custom image
2. **Created** an image repository in Snowflake
3. **Reviewed** the Docker build/push workflow for Snowflake's registry
4. **Submitted** the same training script via standard `submit_directory` for comparison
5. **Compared** the two approaches for different use cases

### Key Takeaways

| Takeaway | Detail |
|----------|--------|
| **Same API** | `submit_directory` works with both approaches |
| **Snowflake image registry** | Built-in, no external registry needed |
| **Docker = production** | Exact reproducibility and faster cold starts |
| **requirements.txt = development** | Zero friction for prototyping |

---

**Next →** [03_distributed_training.ipynb](03_distributed_training.ipynb) — Distribute training across multiple nodes